# **Bài 1**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Tải dữ liệu và chia tập train/test
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Chuẩn hóa dữ liệu đầu vào
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Thực nghiệm với nhiều giá trị K và các độ đo khoảng cách (metric)
metrics = ['euclidean', 'manhattan', 'minkowski']
k_values = range(1, 16)

results = []
for metric in metrics:
    accs = []
    for k in k_values:
        # Với metric='minkowski', có thể đặt p=3 (hoặc mặc định p=2)
        kwargs = {'n_neighbors': k, 'metric': metric}
        if metric == 'minkowski':
            kwargs['p'] = 3

        model = KNeighborsClassifier(**kwargs)
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        acc = accuracy_score(y_test, y_pred)
        accs.append(acc)
        results.append({'metric': metric, 'K': k, 'accuracy': acc})

df_results = pd.DataFrame(results)

# Tìm cấu hình tốt nhất
best_row = df_results.sort_values(by=['accuracy', 'K'], ascending=[False, True]).iloc[0]
print(f"Cấu hình tốt nhất: metric = {best_row['metric']}, K = {int(best_row['K'])}, Accuracy = {best_row['accuracy']:.4f}")

# Vẽ biểu đồ so sánh K theo từng metric
plt.figure(figsize=(9, 5))
for metric in metrics:
    sub = df_results[df_results['metric'] == metric]
    plt.plot(sub['K'], sub['accuracy'], marker='o', label=f"Metric: {metric}")

plt.title("Độ chính xác KNN theo giá trị K và độ đo khoảng cách (Iris - đã chuẩn hóa)")
plt.xlabel("Giá trị K")
plt.ylabel("Accuracy")
plt.xticks(k_values)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

# Đánh giá chi tiết mô hình tối ưu
best_k = int(best_row['K'])
best_metric = best_row['metric']
best_model = KNeighborsClassifier(
    n_neighbors=best_k,
    metric=best_metric,
    p=3 if best_metric == 'minkowski' else 2
)
best_model.fit(X_train_scaled, y_train)
y_pred_best = best_model.predict(X_test_scaled)

print("\nBáo cáo phân loại của mô hình tốt nhất:")
print(classification_report(y_test, y_pred_best, target_names=iris.target_names))

conf_matrix = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(6, 5))
sns.heatmap(
    conf_matrix, annot=True, fmt="d", cmap="YlGnBu",
    xticklabels=iris.target_names, yticklabels=iris.target_names
)
plt.xlabel("Dự đoán")
plt.ylabel("Thực tế")
plt.title(f"Ma trận nhầm lẫn KNN (K={best_k}, metric={best_metric})")
plt.show()

# **Bài 2**

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 1. Tải tập dữ liệu Boston Housing (có dự phòng nếu load_boston bị gỡ bỏ)
def load_boston_data():
    try:
        from sklearn.datasets import load_boston
        boston = load_boston()
        return boston.data, boston.target
    except ImportError:
        url = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"
        try:
            df = pd.read_csv(url)
            X = df.drop(columns=['medv']).values
            y = df['medv'].values
            return X, y
        except Exception:
            from sklearn.datasets import make_regression
            return make_regression(n_samples=506, n_features=13, noise=15.0, random_state=42)

X, y = load_boston_data()

# 2. Phân chia dữ liệu thành tập huấn luyện và tập kiểm tra
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Chuẩn hóa dữ liệu để các đặc trưng ở cùng một khoảng giá trị
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4 & 5. Áp dụng KNN Regressor với các giá trị k = 3, 5, 7, 9 và tính MSE, R^2
k_list = [3, 5, 7, 9]
rows = []

for k in k_list:
    knn_reg = KNeighborsRegressor(n_neighbors=k)
    knn_reg.fit(X_train_scaled, y_train)
    y_pred = knn_reg.predict(X_test_scaled)

    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    rows.append({'k': k, 'MSE': mse, 'R2_score': r2})

df_knn_reg = pd.DataFrame(rows)
print("Kết quả đánh giá KNN Regressor trên tập dữ liệu giá nhà:")
print(df_knn_reg.to_string(index=False))

Kết quả đánh giá KNN Regressor trên tập dữ liệu giá nhà:
 k       MSE  R2_score
 3 19.404281  0.735398
 5 20.605529  0.719017
 7 23.415496  0.680700
 9 24.009424  0.672601


# **Bài 3**

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

iris = load_iris()
X, y = iris.data, iris.target

# Các tỷ lệ test_size tương ứng với 90/10, 80/20, 70/30, 60/40
splits = [
    ("90% Train / 10% Test", 0.1),
    ("80% Train / 20% Test", 0.2),
    ("70% Train / 30% Test", 0.3),
    ("60% Train / 40% Test", 0.4),
]

comparison = []

for split_name, test_sz in splits:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_sz, random_state=42, stratify=y
    )

    # Trường hợp 1: Không chuẩn hóa
    svm_raw = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
    svm_raw.fit(X_train, y_train)
    acc_raw = accuracy_score(y_test, svm_raw.predict(X_test))

    # Trường hợp 2: Có chuẩn hóa StandardScaler
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)

    svm_scaled = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
    svm_scaled.fit(X_train_sc, y_train)
    y_pred_sc = svm_scaled.predict(X_test_sc)
    acc_scaled = accuracy_score(y_test, y_pred_sc)

    comparison.append({
        'Cách chia dữ liệu': split_name,
        'Accuracy (Chưa chuẩn hóa)': round(acc_raw, 4),
        'Accuracy (Đã chuẩn hóa)': round(acc_scaled, 4)
    })

    print(f"=== {split_name} (Có chuẩn hóa) ===")
    print(classification_report(y_test, y_pred_sc, target_names=iris.target_names, digits=4))

df_comp = pd.DataFrame(comparison)
print("Bảng so sánh kết quả giữa các cách chia dữ liệu:")
print(df_comp.to_string(index=False))

=== 90% Train / 10% Test (Có chuẩn hóa) ===
              precision    recall  f1-score   support

      setosa     1.0000    1.0000    1.0000         5
  versicolor     1.0000    0.8000    0.8889         5
   virginica     0.8333    1.0000    0.9091         5

    accuracy                         0.9333        15
   macro avg     0.9444    0.9333    0.9327        15
weighted avg     0.9444    0.9333    0.9327        15

=== 80% Train / 20% Test (Có chuẩn hóa) ===
              precision    recall  f1-score   support

      setosa     1.0000    1.0000    1.0000        10
  versicolor     1.0000    0.9000    0.9474        10
   virginica     0.9091    1.0000    0.9524        10

    accuracy                         0.9667        30
   macro avg     0.9697    0.9667    0.9666        30
weighted avg     0.9697    0.9667    0.9666        30

=== 70% Train / 30% Test (Có chuẩn hóa) ===
              precision    recall  f1-score   support

      setosa     1.0000    1.0000    1.0000        

# **Bài 4**

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

# 1. Tải dữ liệu (dùng lại hàm từ Bài 2)
def load_boston_data():
    try:
        from sklearn.datasets import load_boston
        boston = load_boston()
        return boston.data, boston.target
    except ImportError:
        url = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"
        try:
            df = pd.read_csv(url)
            X = df.drop(columns=['medv']).values
            y = df['medv'].values
            return X, y
        except Exception:
            from sklearn.datasets import make_regression
            return make_regression(n_samples=506, n_features=13, noise=15.0, random_state=42)

X, y = load_boston_data()

# 2. Chia dữ liệu và chuẩn hóa
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Huấn luyện và đánh giá SVR với kernel='linear' và kernel='rbf'
kernels = ['linear', 'rbf']
svr_results = []

for kernel in kernels:
    svr_model = SVR(kernel=kernel, C=10.0)
    svr_model.fit(X_train_scaled, y_train)
    y_pred = svr_model.predict(X_test_scaled)

    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    svr_results.append({
        'Kernel': kernel,
        'MSE': mse,
        'R2_score': r2
    })

df_svr = pd.DataFrame(svr_results)
print("Kết quả đánh giá mô hình SVR trên tập dữ liệu giá nhà:")
print(df_svr.to_string(index=False))

Kết quả đánh giá mô hình SVR trên tập dữ liệu giá nhà:
Kernel       MSE  R2_score
linear 28.785637  0.607471
   rbf 12.738192  0.826298
